# Divide & Conquer

Notes and runnable examples on **divide & conquer** — the algorithm-design paradigm that **splits** a problem into smaller independent subproblems, **solves** each (usually by recursing), and **combines** their answers into the whole. The angle here is the *recurrence*: every D&C algorithm is summarized by a recurrence like `T(n) = a·T(n/b) + f(n)`, and the **Master Theorem** reads its complexity straight off `a`, `b`, and `f`. We'll state the theorem, then build four classics from scratch — Karatsuba multiplication, counting inversions, and maximum-subarray — each **proven correct against a brute-force oracle** over thousands of seeded-random cases.

This is the paradigm behind merge sort (the **sorting** notebook), binary search (the **searching** notebook), and quickselect (the **selection** notebook); it leans entirely on the call stack from the **recursion & backtracking** notebook.

**Contents**

1. **The paradigm & the recurrence** — split / solve / combine, and the **Master Theorem**
2. **Karatsuba multiplication** — O(n^1.585) big-integer multiply
3. **Counting inversions** — a merge-sort variant in O(n log n)
4. **Maximum subarray** — the cross-boundary combine
5. **When to use / how to recognize D&C + cheat-sheet**

## 1. The paradigm & the recurrence

Divide & conquer is three steps, applied recursively until the problem is trivial:

1. **Divide** — break the input into `a` subproblems, each of size `n/b`.
2. **Conquer** — solve each subproblem (recurse; the base case handles size-1 directly).
3. **Combine** — stitch the sub-answers into the answer for the whole, doing `f(n)` work.

The cost is captured by a **recurrence**:

$$T(n) = a \cdot T(n/b) + f(n)$$

where `a` = number of subproblems, `b` = shrink factor, and `f(n)` = divide + combine work. The whole game is balancing the **recursion tree** (`a^depth` leaves) against the per-level combine cost. The **Master Theorem** solves this whole family at once by comparing `f(n)` to the *watershed* function `n^(log_b a)` — the cost if all the work lived in the leaves:

| Case | Condition (compare `f(n)` to `n^log_b(a)`) | Result | Intuition |
|:---:|---|:---:|---|
| **1** | `f(n) = O(n^(log_b a − ε))` — leaves dominate | `T(n) = Θ(n^(log_b a))` | work piles up at the bottom |
| **2** | `f(n) = Θ(n^(log_b a) · log^k n)` — balanced | `T(n) = Θ(n^(log_b a) · log^(k+1) n)` | every level costs the same → ×log n |
| **3** | `f(n) = Ω(n^(log_b a + ε))` (+ regularity) — root dominates | `T(n) = Θ(f(n))` | the top combine swamps everything |

The canonical instances:

| Algorithm | Recurrence | `a` | `b` | `log_b a` | `f(n)` | Case | Result |
|---|---|:---:|:---:|:---:|:---:|:---:|---|
| **merge sort** | `2T(n/2)+O(n)` | 2 | 2 | 1 | `n` | 2 | `O(n log n)` |
| **binary search** | `1T(n/2)+O(1)` | 1 | 2 | 0 | `1` | 2 | `O(log n)` |
| **Karatsuba** | `3T(n/2)+O(n)` | 3 | 2 | 1.585 | `n` | 1 | `O(n^1.585)` |
| **Strassen** | `7T(n/2)+O(n²)` | 7 | 2 | 2.807 | `n²` | 1 | `O(n^2.807)` |

Let's read those exponents straight off the formula, and confirm the merge-sort recurrence is genuinely the Case-2 `n log n` by *counting the work* the recursion actually does:

In [1]:
import math

def master_exponent(a, b):
    return math.log(a, b)                 # the watershed exponent log_b(a)

for name, a, b in [("merge sort", 2, 2), ("binary search", 1, 2),
                   ("Karatsuba", 3, 2), ("Strassen", 7, 2)]:
    print(f"  {name:14} a={a} b={b} -> n^(log_b a) = n^{master_exponent(a, b):.3f}")

# Prove merge sort's recurrence really sums to ~n*log2(n) by instrumenting the combine work.
def merge_sort_work(a):
    """Returns (sorted_list, total_combine_comparisons) — the f(n)=O(n) per level adds up."""
    if len(a) <= 1:
        return a, 0
    mid = len(a) // 2
    left, wl = merge_sort_work(a[:mid])
    right, wr = merge_sort_work(a[mid:])
    out, work, i, j = [], wl + wr, 0, 0
    while i < len(left) and j < len(right):
        work += 1                          # one comparison = one unit of combine work
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:]); out.extend(right[j:])
    return out, work

import random
random.seed(0)
print(f"\n{'n':>8}{'combine work':>14}{'n*log2(n)':>14}{'ratio':>8}")
for n in (1000, 4000, 16000, 64000):
    data = [random.random() for _ in range(n)]
    _, work = merge_sort_work(data)
    model = n * math.log2(n)
    print(f"{n:>8}{work:>14}{model:>14.0f}{work/model:>8.2f}")

  merge sort     a=2 b=2 -> n^(log_b a) = n^1.000
  binary search  a=1 b=2 -> n^(log_b a) = n^0.000
  Karatsuba      a=3 b=2 -> n^(log_b a) = n^1.585
  Strassen       a=7 b=2 -> n^(log_b a) = n^2.807

       n  combine work     n*log2(n)   ratio
    1000          8721          9966    0.88
    4000         42767         47863    0.89
   16000        203293        223453    0.91
   64000        941410       1021810    0.92


The measured combine work tracks `n·log₂n` with a steady sub-1 ratio (each merge level does ≤ n−1 comparisons across `log₂n` levels) — that's the Master Theorem's **Case 2** in the flesh: every level of the recursion tree costs Θ(n), and there are Θ(log n) levels, so the total is Θ(n log n). Binary search is the degenerate D&C — it *discards* one of its two subproblems instead of combining (`a=1`), collapsing the tree to a single path of length log n. The rest of this notebook builds three D&C algorithms where the **combine** step is where the cleverness lives.

## 2. Karatsuba multiplication — O(n^1.585)

Multiplying two n-digit integers the schoolbook way is **O(n²)**: every digit of one times every digit of the other. **Karatsuba** (1962) was the first to beat that, and it's pure divide & conquer. Split each number into high and low halves (`x = x₁·Bᵐ + x₀`, where B is the base and m the split point):

$$x \cdot y = x_1 y_1 \cdot B^{2m} + (x_1 y_0 + x_0 y_1) \cdot B^{m} + x_0 y_0$$

The naive split needs **four** half-size multiplies (`x₁y₁, x₁y₀, x₀y₁, x₀y₀`) → `4T(n/2)+O(n)` → still O(n²) by the Master Theorem (`log₂4 = 2`). Karatsuba's trick: the middle term `x₁y₀ + x₀y₁` can be recovered from the other two products plus **one** extra multiply, because

$$(x_1 + x_0)(y_1 + y_0) = x_1 y_1 + (x_1 y_0 + x_0 y_1) + x_0 y_0$$

so `x₁y₀ + x₀y₁ = (x₁+x₀)(y₁+y₀) − x₁y₁ − x₀y₀`. That's **three** half-size multiplies instead of four → `3T(n/2)+O(n)` → **O(n^log₂3) = O(n^1.585)**. We split on **bits** (base-2), so `Bᵐ` is just a left shift `<< m` — the additions and shifts are the cheap O(n) combine.

CPython's own `int` does exactly this: above a threshold (`KARATSUBA_CUTOFF`, ~70 internal 30-bit *digits*, i.e. roughly 600+ bit numbers) the big-integer multiply in `Objects/longobject.c` switches from schoolbook to Karatsuba. So Python's `*` operator *is* a divide-and-conquer algorithm on large ints — see the **bit manipulation** notebook for the `PyLong` digit layout (`sys.int_info` reports 30-bit digits).

In [2]:
import random
random.seed(1)

def karatsuba(x, y):
    # base case: small enough that CPython's own multiply is cheaper than recursing
    if x < (1 << 64) or y < (1 << 64):
        return x * y
    n = max(x.bit_length(), y.bit_length())
    m = n // 2                               # split point, in bits
    high_x, low_x = x >> m, x & ((1 << m) - 1)   # x = high_x*2^m + low_x
    high_y, low_y = y >> m, y & ((1 << m) - 1)
    z0 = karatsuba(low_x,  low_y)                       # x0*y0
    z2 = karatsuba(high_x, high_y)                      # x1*y1
    z1 = karatsuba(low_x + high_x, low_y + high_y) - z0 - z2   # the saved middle term
    return (z2 << (2 * m)) + (z1 << m) + z0             # shifts + adds = O(n) combine

# PROOF: matches Python's built-in * on many random big integers of varied sizes
for _ in range(3000):
    x = random.getrandbits(random.randint(1, 2048))
    y = random.getrandbits(random.randint(1, 2048))
    assert karatsuba(x, y) == x * y
print("karatsuba == (x * y) over 3000 random big-int pairs  [verified]")

import sys
print("PyLong digits are", sys.int_info.bits_per_digit, "bits wide (sys.int_info)")

karatsuba == (x * y) over 3000 random big-int pairs  [verified]
PyLong digits are 30 bits wide (sys.int_info)


In [3]:
import timeit, random
random.seed(2)

# Schoolbook O(n^2) in pure Python (base-2^32 limbs) to race against our Karatsuba.
def schoolbook(x, y):
    B = 32; MASK = (1 << B) - 1
    xs, ys = [], []
    t = x
    while t: xs.append(t & MASK); t >>= B
    t = y
    while t: ys.append(t & MASK); t >>= B
    xs = xs or [0]; ys = ys or [0]
    res = [0] * (len(xs) + len(ys))
    for i, xi in enumerate(xs):
        carry = 0
        for j, yj in enumerate(ys):
            cur = res[i + j] + xi * yj + carry
            res[i + j] = cur & MASK
            carry = cur >> B
        res[i + len(ys)] += carry
    out = 0
    for limb in reversed(res):
        out = (out << B) | limb
    return out

x = random.getrandbits(4096); y = random.getrandbits(4096)
assert karatsuba(x, y) == schoolbook(x, y) == x * y

tk = timeit.timeit(lambda: karatsuba(x, y),  number=300) / 300
ts = timeit.timeit(lambda: schoolbook(x, y), number=300) / 300
tb = timeit.timeit(lambda: x * y,            number=3000) / 3000
print("4096-bit multiply:")
print(f"  karatsuba  (pure Python, O(n^1.585)): {tk*1e6:8.1f} us")
print(f"  schoolbook (pure Python, O(n^2))    : {ts*1e6:8.1f} us   -> {ts/tk:4.1f}x slower")
print(f"  x * y      (CPython C, Karatsuba)   : {tb*1e6:8.2f} us   -> the real thing")

4096-bit multiply:
  karatsuba  (pure Python, O(n^1.585)):    220.1 us
  schoolbook (pure Python, O(n^2))    :   1658.4 us   ->  7.5x slower
  x * y      (CPython C, Karatsuba)   :     8.31 us   -> the real thing


Two lessons at once. **Asymptotics win:** our Karatsuba is several times faster than pure-Python schoolbook at 4096 bits, and the gap *widens* with size (`n^1.585` vs `n²`). **But constants matter too:** CPython's built-in `*` — itself a Karatsuba implementation, just in C with a tight schoolbook base case — is orders of magnitude faster than any pure-Python version. The paradigm is the same; the implementation language sets the constant factor (the recurring theme of the **CPython cost model** notebook). You'd never hand-roll bignum multiply in Python — but knowing `*` is O(n^1.585), not O(n²), explains why multiplying 10,000-digit numbers is so much faster than you'd fear.

## 3. Counting inversions — O(n log n)

An **inversion** is a pair of indices `i < j` with `a[i] > a[j]` — an out-of-order pair. The number of inversions measures how unsorted an array is: a sorted array has 0, a reverse-sorted array has the maximum `n(n−1)/2`. (It's exactly the number of adjacent swaps bubble sort would make, and underlies rank-correlation statistics like Kendall's tau.)

The brute force is **O(n²)**: check every pair. The divide & conquer insight piggybacks on **merge sort**: while merging two sorted halves, every time you pull an element from the *right* half before the left half is exhausted, that right element is smaller than **all remaining** left-half elements — so it forms an inversion with each of them. Count `len(left) − i` in one shot. The split and recurse are merge sort's; the combine does the counting essentially for free, giving the merge-sort recurrence `2T(n/2)+O(n)` = **O(n log n)**.

In [4]:
def count_inversions(a):
    """Returns the number of pairs i<j with a[i] > a[j], in O(n log n)."""
    def sort_and_count(arr):
        if len(arr) <= 1:
            return arr, 0
        mid = len(arr) // 2
        left,  cl = sort_and_count(arr[:mid])
        right, cr = sort_and_count(arr[mid:])
        merged, count, i, j = [], cl + cr, 0, 0       # inversions within each half carry up
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i]); i += 1
            else:
                merged.append(right[j]); j += 1
                count += len(left) - i                # right[j] beats every remaining left elem
        merged.extend(left[i:]); merged.extend(right[j:])
        return merged, count
    return sort_and_count(a)[1]

def count_inversions_brute(a):                        # the O(n^2) oracle
    return sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if a[i] > a[j])

# PROOF: matches the brute-force oracle over thousands of random arrays (incl. duplicates & empties)
import random
random.seed(3)
for _ in range(2000):
    n = random.randint(0, 80)
    a = [random.randint(-6, 6) for _ in range(n)]     # small range -> lots of ties
    assert count_inversions(a) == count_inversions_brute(a)
print("count_inversions == brute force over 2000 random arrays  [verified]")

print("sorted [1,2,3,4,5]   ->", count_inversions([1, 2, 3, 4, 5]), "inversions")
print("reverse [5,4,3,2,1]  ->", count_inversions([5, 4, 3, 2, 1]),
      "inversions  (= n(n-1)/2 = max)")

count_inversions == brute force over 2000 random arrays  [verified]
sorted [1,2,3,4,5]   -> 0 inversions
reverse [5,4,3,2,1]  -> 10 inversions  (= n(n-1)/2 = max)


In [5]:
import timeit, random
random.seed(4)

# The O(n log n) shape: each doubling of n should roughly double the time (n log n ~ linear-ish),
# whereas the O(n^2) brute force should *quadruple*. Watch both ratios.
print(f"{'n':>6}{'DnC ms':>10}{'DnC x':>8}{'brute ms':>12}{'brute x':>9}")
prev_d = prev_b = None
for n in (500, 1000, 2000, 4000):
    a = [random.randint(0, 10**6) for _ in range(n)]
    td = timeit.timeit(lambda: count_inversions(a),       number=30) / 30 * 1000
    tb = timeit.timeit(lambda: count_inversions_brute(a), number=5)  / 5  * 1000
    rd = f"{td/prev_d:.1f}" if prev_d else "  -"
    rb = f"{tb/prev_b:.1f}" if prev_b else "  -"
    print(f"{n:>6}{td:>10.2f}{rd:>8}{tb:>12.1f}{rb:>9}")
    prev_d, prev_b = td, tb

     n    DnC ms   DnC x    brute ms  brute x
   500      0.33       -         2.7        -
  1000      0.69     2.0        11.4      4.2


  2000      1.55     2.3        55.3      4.9


  4000      3.31     2.1       175.1      3.2


The pattern is unmistakable: doubling `n` roughly **doubles** the D&C time (the linearithmic curve looks nearly linear over a 8× range) but **quadruples** the brute force — the signature of O(n log n) vs O(n²). On 4000 elements the D&C is already an order of magnitude faster, and the gap explodes from there. The same merge-and-count trick computes any "out-of-order pair" statistic; swap the comparison and you count *non-*inversions, or weight them, all in the same linearithmic pass.

## 4. Maximum subarray — the cross-boundary combine

The **maximum-subarray** problem: find the contiguous slice with the largest sum (classic in finance — best buy/sell window). With negative numbers it's non-trivial; brute force is O(n²) (or O(n³) naively). The divide & conquer solution is the textbook illustration of a *combine step that does real work*:

- **Divide** at the midpoint into left and right halves.
- **Conquer** — the best subarray is *entirely in the left*, *entirely in the right*, or **crosses the midpoint**.
- **Combine** — the first two come from recursion; for the crossing case, walk *outward from the center* to find the best left-extending and best right-extending sums, and join them. That scan is O(n).

Recurrence `2T(n/2)+O(n)` = **O(n log n)**. It's beaten by **Kadane's algorithm** — a one-pass O(n) dynamic-programming sweep — but the D&C version is the cleaner illustration of the paradigm, and it generalizes to settings (like the maximum-subrectangle or certain segment-tree queries) where no linear sweep exists. We prove our D&C against *both* Kadane and brute force.

In [6]:
def max_subarray_dc(a):
    """Largest sum of any non-empty contiguous subarray, via divide & conquer."""
    def rec(lo, hi):
        if lo == hi:
            return a[lo]                          # base case: single element
        mid = (lo + hi) // 2
        best_left  = rec(lo, mid)                 # entirely in the left half
        best_right = rec(mid + 1, hi)             # entirely in the right half
        # crossing the midpoint: best sum ending at mid, extended best sum starting at mid+1
        s, left_part = 0, float("-inf")
        for i in range(mid, lo - 1, -1):
            s += a[i]; left_part = max(left_part, s)
        s, right_part = 0, float("-inf")
        for i in range(mid + 1, hi + 1):
            s += a[i]; right_part = max(right_part, s)
        return max(best_left, best_right, left_part + right_part)   # the combine
    return rec(0, len(a) - 1)

def kadane(a):                                    # the O(n) one-pass DP (the thing to actually use)
    best = cur = a[0]
    for x in a[1:]:
        cur = max(x, cur + x)                     # extend or restart
        best = max(best, cur)
    return best

def max_subarray_brute(a):                        # O(n^2) oracle
    return max(sum(a[i:j]) for i in range(len(a)) for j in range(i + 1, len(a) + 1))

# PROOF: D&C agrees with BOTH Kadane and the brute-force oracle over thousands of arrays
import random
random.seed(5)
for _ in range(3000):
    n = random.randint(1, 60)
    a = [random.randint(-12, 12) for _ in range(n)]      # mixed signs make it interesting
    expected = max_subarray_brute(a)
    assert max_subarray_dc(a) == expected == kadane(a)
print("max_subarray_dc == kadane == brute force over 3000 random arrays  [verified]")

demo = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
print("example", demo, "-> max subarray sum =", max_subarray_dc(demo), "(the slice [4,-1,2,1])")

max_subarray_dc == kadane == brute force over 3000 random arrays  [verified]
example [-2, 1, -3, 4, -1, 2, 1, -5, 4] -> max subarray sum = 6 (the slice [4,-1,2,1])


All three agree on every case. The D&C is O(n log n) where Kadane is O(n), so in 1-D you'd reach for Kadane — but the *cross-boundary combine* idea is the transferable lesson: when a problem's answer can lie in the left, the right, or **straddling the split**, you handle the straddle explicitly in the combine step. That's also the heart of the **closest pair of points** algorithm — split the points by x-coordinate, recurse on each side, then check only the narrow strip straddling the divide — which gets the closest pair in O(n log n) instead of the O(n²) all-pairs check; see the **computational geometry** notebook for that one.

## 5. When to use / how to recognize D&C + cheat-sheet

**Recognize a divide-and-conquer opportunity when:**

- the problem **decomposes into independent subproblems of the same form** (so you can recurse), and
- there's a way to **combine** sub-answers more cheaply than re-solving from scratch, and
- the subproblems **don't overlap** — if they *do*, you want memoization / **dynamic programming** instead (the recursion → DP story from the **recursion & backtracking** notebook). Karatsuba's three products are disjoint; Fibonacci's two recursive calls overlap, which is why it needs caching, not plain D&C.

| You see... | Reach for | Recurrence | Result |
|---|---|---|---|
| Sort a sequence | merge sort (the **sorting** notebook) | `2T(n/2)+O(n)` | O(n log n) |
| Search a *sorted* sequence | binary search (the **searching** notebook) | `1T(n/2)+O(1)` | O(log n) |
| k-th smallest without full sort | quickselect (the **selection** notebook) | `T(n)+O(n)` avg | O(n) avg |
| Multiply huge integers | Karatsuba (here / CPython's `int`) | `3T(n/2)+O(n)` | O(n^1.585) |
| Count out-of-order pairs | merge-and-count (here) | `2T(n/2)+O(n)` | O(n log n) |
| Best contiguous-sum slice | Kadane (1-D) / D&C combine (here) | `2T(n/2)+O(n)` | O(n) / O(n log n) |
| Closest pair of points | strip combine (**computational geometry**) | `2T(n/2)+O(n)` | O(n log n) |
| Overlapping subproblems | **don't** — use memoization / DP | — | — |

**Master Theorem cheat-sheet** — for `T(n) = a·T(n/b) + f(n)`, compare `f(n)` to the watershed `n^(log_b a)`:

| Case | When | `T(n)` | Example |
|:---:|---|:---:|---|
| **1** (leaf-heavy) | `f(n)` grows slower: `O(n^(log_b a − ε))` | `Θ(n^(log_b a))` | Karatsuba `3T(n/2)+O(n)` → `Θ(n^1.585)` |
| **2** (balanced) | `f(n) = Θ(n^(log_b a)·log^k n)` | `Θ(n^(log_b a)·log^(k+1) n)` | merge sort `2T(n/2)+O(n)` → `Θ(n log n)` |
| **3** (root-heavy) | `f(n)` grows faster: `Ω(n^(log_b a + ε))` (+ regularity) | `Θ(f(n))` | `2T(n/2)+O(n²)` → `Θ(n²)` |

| Concept | Key fact |
|---|---|
| The shape | **divide** into `a` parts of size `n/b` → **conquer** (recurse) → **combine** in `f(n)` |
| Where the cleverness lives | usually the **combine** step (inversions counted during merge; cross-sum; Karatsuba's 3-mult identity) |
| Master Theorem | read complexity off `a`, `b`, `f(n)` by comparing `f(n)` to `n^(log_b a)` (3 cases) |
| D&C vs DP | D&C needs **disjoint** subproblems; **overlapping** ⇒ dynamic programming / memoization |
| Constant factors | the asymptotic win is real, but CPython's C builtins still dwarf pure-Python D&C (see Karatsuba) |
| Stack depth | recursion depth is `O(log n)` for balanced splits — well under CPython's limit |